# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Method choice: This is a binary classification and ranking problem because the target indicates whether a page is declining. I will start with Logistic Regression because it provides an interpretable probability score for ranking pages and gives a simple learned benchmark. If it does not beat the Week-4 rule baseline, I will not add complexity unnecessarily; if useful, I will compare a Random Forest to test whether non-linear relationships improve the ranking.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Split design: I used an 80/20 client-grouped train/test split. Pages from the same client are kept entirely within either the training or testing set. This prevents client-specific patterns from leaking across the split and gives a more honest estimate of how the model may perform on unseen clients. The split contains 25 training clients and 7 testing clients, with zero client overlap.

In [19]:
import pandas as pd

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully")
print("Shape:", df.shape)

Dataset loaded successfully
Shape: (30000, 44)


In [20]:
# =========================================================
# SECTION 2 — SPLIT DESIGN
# =========================================================

import pandas as pd
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit


# ---------------------------------------------------------
# 1. Load the raw dataset
# ---------------------------------------------------------

DATA_PATH = Path(
    "/content/FlyRank-ML-Internship/data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Shape:", df.shape)


# ---------------------------------------------------------
# 2. Create the target label
# ---------------------------------------------------------
# The raw dataset does not contain is_declining_label.
#
# According to the data dictionary:
# trend_direction == "down"  -> declining page (1)
# anything else              -> not declining (0)
#
# IMPORTANT:
# trend_direction is used to CREATE the label.
# Therefore, trend_direction must NOT be used as a model feature.

df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)


print("\nTARGET LABEL")
print("=" * 50)

print(df["is_declining_label"].value_counts())

print(
    "Overall decline rate:",
    round(df["is_declining_label"].mean() * 100, 2),
    "%"
)


# ---------------------------------------------------------
# 3. Define target and grouping variable
# ---------------------------------------------------------

y = df["is_declining_label"]

# client_id is used ONLY to create the grouped split.
# It will NOT be used as a model feature.
groups = df["client_id"]


# ---------------------------------------------------------
# 4. Create a client-grouped train/test split
# ---------------------------------------------------------
#
# 80% of clients -> training
# 20% of clients -> testing
#
# random_state=42 makes the split reproducible.

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(df, y, groups=groups)
)


# ---------------------------------------------------------
# 5. Create train/test datasets
# ---------------------------------------------------------

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

y_train = train_df["is_declining_label"]
y_test = test_df["is_declining_label"]


# ---------------------------------------------------------
# 6. Verify client separation
# ---------------------------------------------------------

train_clients = set(train_df["client_id"])
test_clients = set(test_df["client_id"])

overlap = train_clients.intersection(test_clients)


print("\nTRAIN / TEST SPLIT")
print("=" * 50)

print("Total clients:", df["client_id"].nunique())
print("Training clients:", len(train_clients))
print("Testing clients:", len(test_clients))

print()

print("Training pages:", len(train_df))
print("Testing pages:", len(test_df))

print()

print("Client overlap:", len(overlap))


if len(overlap) == 0:
    print("✓ VALID: No client appears in both training and testing.")
else:
    print("✗ ERROR: Client leakage detected!")
    print("Overlapping clients:", overlap)


# ---------------------------------------------------------
# 7. Compare target rates
# ---------------------------------------------------------

print("\nTARGET DISTRIBUTION")
print("=" * 50)

print(
    "Overall decline rate:",
    round(y.mean() * 100, 2),
    "%"
)

print(
    "Training decline rate:",
    round(y_train.mean() * 100, 2),
    "%"
)

print(
    "Testing decline rate:",
    round(y_test.mean() * 100, 2),
    "%"
)


# ---------------------------------------------------------
# 8. Final verification
# ---------------------------------------------------------

print("\nSECTION 2 CHECK")
print("=" * 50)

if len(overlap) == 0:
    print("✓ Client-grouped split is valid.")
    print("✓ No client appears in both train and test.")
    print("✓ client_id is being used for grouping only.")
    print("✓ Target is is_declining_label.")
    print("✓ trend_direction will not be used as a model feature.")
    print("✓ trend_pct will not be used as a model feature.")
else:
    print("✗ Split needs to be fixed because clients overlap.")

Dataset loaded successfully.
Shape: (30000, 44)

TARGET LABEL
is_declining_label
1    16262
0    13738
Name: count, dtype: int64
Overall decline rate: 54.21 %

TRAIN / TEST SPLIT
Total clients: 32
Training clients: 25
Testing clients: 7

Training pages: 23837
Testing pages: 6163

Client overlap: 0
✓ VALID: No client appears in both training and testing.

TARGET DISTRIBUTION
Overall decline rate: 54.21 %
Training decline rate: 55.01 %
Testing decline rate: 51.1 %

SECTION 2 CHECK
✓ Client-grouped split is valid.
✓ No client appears in both train and test.
✓ client_id is being used for grouping only.
✓ Target is is_declining_label.
✓ trend_direction will not be used as a model feature.
✓ trend_pct will not be used as a model feature.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [21]:
# ============================================================
# SECTION 3 — STEP 1: Prepare model features
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Create the target label
# ------------------------------------------------------------
# A page is declining when trend_direction == "down".
# IMPORTANT: trend_direction itself will NOT be given to the model.

df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

# ------------------------------------------------------------
# 2. Create the log-transformed traffic features
# ------------------------------------------------------------
# Traffic numbers are heavy-tailed, so log1p reduces the
# influence of extremely large pages.

for col in [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ai_sessions_90d"
]:
    df[f"log_{col}"] = np.log1p(
        pd.to_numeric(df[col], errors="coerce").fillna(0)
    )

# ------------------------------------------------------------
# 3. Define the features we are allowed to use
# ------------------------------------------------------------

MODEL_NUMERIC_FEATURES = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "log_impressions_90d",
    "log_clicks_90d",
    "log_sessions_90d",
    "log_ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

MODEL_CATEGORICAL_FEATURES = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "impression_tier",
    "position_tier",
]

MODEL_FEATURES = (
    MODEL_NUMERIC_FEATURES +
    MODEL_CATEGORICAL_FEATURES
)

# ------------------------------------------------------------
# 4. Check that every feature exists
# ------------------------------------------------------------

missing_features = [
    col for col in MODEL_FEATURES
    if col not in df.columns
]

print("MODEL FEATURE CHECK")
print("=" * 50)

if missing_features:
    print("Missing features:", missing_features)
else:
    print("✓ All model features are present.")

# ------------------------------------------------------------
# 5. Re-create the SAME client-grouped split
# ------------------------------------------------------------
# We do this after creating the features so X_train/X_test
# contain the actual model-ready columns.

from sklearn.model_selection import GroupShuffleSplit

X = df[MODEL_FEATURES].copy()
y = df["is_declining_label"].copy()
groups = df["client_id"].copy()

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

train_clients = set(df.iloc[train_idx]["client_id"])
test_clients = set(df.iloc[test_idx]["client_id"])

# ------------------------------------------------------------
# 6. Final verification
# ------------------------------------------------------------

print()
print("MODEL DATA")
print("=" * 50)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Number of features:", len(MODEL_FEATURES))

print()
print("Client overlap:", len(train_clients & test_clients))

print()
print("Training decline rate:",
      round(y_train.mean() * 100, 2), "%")

print("Testing decline rate:",
      round(y_test.mean() * 100, 2), "%")

print()
print("LEAKAGE CHECK")
print("=" * 50)

for forbidden in [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "content_id",
    "client_id"
]:
    if forbidden in MODEL_FEATURES:
        print("✗ LEAKAGE:", forbidden)
    else:
        print("✓ Not a model feature:", forbidden)

MODEL FEATURE CHECK
✓ All model features are present.

MODEL DATA
Training rows: 23837
Testing rows: 6163
Number of features: 27

Client overlap: 0

Training decline rate: 55.01 %
Testing decline rate: 51.1 %

LEAKAGE CHECK
✓ Not a model feature: trend_direction
✓ Not a model feature: trend_pct
✓ Not a model feature: is_declining_label
✓ Not a model feature: content_id
✓ Not a model feature: client_id


In [22]:
# ============================================================
# SECTION 3 — STEP 2: Logistic Regression
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score

# ------------------------------------------------------------
# 1. Preprocessing
# ------------------------------------------------------------

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True
        )),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, MODEL_NUMERIC_FEATURES),
        ("categorical", categorical_pipeline, MODEL_CATEGORICAL_FEATURES),
    ]
)

# ------------------------------------------------------------
# 2. Logistic Regression model
# ------------------------------------------------------------

logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        ),
    ]
)

# ------------------------------------------------------------
# 3. Train ONLY on training clients
# ------------------------------------------------------------

logistic_model.fit(X_train, y_train)

print("Logistic Regression trained successfully.")

# ------------------------------------------------------------
# 4. Predict probabilities on unseen clients
# ------------------------------------------------------------

y_test_probability = logistic_model.predict_proba(
    X_test
)[:, 1]

y_test_prediction = (
    y_test_probability >= 0.5
).astype(int)

# ------------------------------------------------------------
# 5. Standard classification metrics
# ------------------------------------------------------------

accuracy = accuracy_score(
    y_test,
    y_test_prediction
)

precision = precision_score(
    y_test,
    y_test_prediction,
    zero_division=0
)

recall = recall_score(
    y_test,
    y_test_prediction,
    zero_division=0
)

# ------------------------------------------------------------
# 6. Precision@50
# ------------------------------------------------------------

def precision_at_k(y_true, scores, k=50):
    result = pd.DataFrame({
        "actual": np.asarray(y_true),
        "score": np.asarray(scores)
    })

    top_k = result.sort_values(
        "score",
        ascending=False
    ).head(k)

    return top_k["actual"].mean()

model_precision_at_50 = precision_at_k(
    y_test,
    y_test_probability,
    k=50
)

# ------------------------------------------------------------
# 7. Results
# ------------------------------------------------------------

print()
print("LOGISTIC REGRESSION RESULTS")
print("=" * 50)

print("Accuracy:", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print(
    "Precision@50:",
    round(model_precision_at_50, 4)
)

print()
print(
    "Top 50 predicted-decline pages contain",
    round(model_precision_at_50 * 50),
    "actual declining pages out of 50."
)

Logistic Regression trained successfully.

LOGISTIC REGRESSION RESULTS
Accuracy: 0.5846
Precision: 0.5824
Recall: 0.6608
Precision@50: 0.74

Top 50 predicted-decline pages contain 37 actual declining pages out of 50.


In [23]:
# ============================================================
# SECTION 3 — STEP 3A: Week-5 Baseline on SAME Test Clients
# ============================================================

# Get the ORIGINAL dataset rows belonging to the test clients.
# This gives us baseline columns such as impressions_90d.
baseline_test = df.iloc[test_idx].copy()

# ------------------------------------------------------------
# Recreate Assignment 5 baseline rule
#
# Eligible if:
#   1. Page has not been updated for at least 91 days
#   2. Page has at least 300 impressions
#
# Then rank eligible pages by impressions_90d.
# ------------------------------------------------------------

baseline_test["baseline_score"] = np.where(
    (
        (baseline_test["days_since_last_update"] >= 91)
        &
        (baseline_test["impressions_90d"] >= 300)
    ),
    baseline_test["impressions_90d"],
    0
)

# ------------------------------------------------------------
# Rank exactly like Assignment 5
# ------------------------------------------------------------

baseline_test = baseline_test.sort_values(
    "baseline_score",
    ascending=False
).reset_index(drop=True)

# ------------------------------------------------------------
# Top 50
# ------------------------------------------------------------

baseline_top50 = baseline_test.head(50)

baseline_precision_at_50 = (
    baseline_top50["is_declining_label"].mean()
)

baseline_declining_count = (
    baseline_top50["is_declining_label"].sum()
)

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print("BASELINE RESULTS")
print("=" * 50)

print(
    "Baseline Precision@50:",
    round(baseline_precision_at_50, 4)
)

print(
    "Actual declining pages in top 50:",
    int(baseline_declining_count),
    "out of 50"
)

print()

print("Baseline top 5:")
print(
    baseline_top50[
        [
            "content_id",
            "days_since_last_update",
            "impressions_90d",
            "is_declining_label",
            "baseline_score"
        ]
    ].head()
)

BASELINE RESULTS
Baseline Precision@50: 0.3
Actual declining pages in top 50: 15 out of 50

Baseline top 5:
             content_id  days_since_last_update  impressions_90d  \
0  content_5fe46e04994d                     104           517715   
1  content_9532f197bbc8                     104           309192   
2  content_1aa219431528                     104           151541   
3  content_3ad3a781fa91                     104           145044   
4  content_e6e15ac13287                     104           128704   

   is_declining_label  baseline_score  
0                   1          517715  
1                   1          309192  
2                   0          151541  
3                   0          145044  
4                   0          128704  


In [24]:
# ============================================================
# SECTION 3 — RANDOM FOREST WITH CATEGORICAL ENCODING
# ============================================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score

# ------------------------------------------------------------
# Identify numeric and categorical features
# ------------------------------------------------------------

numeric_features = [
    col for col in MODEL_NUMERIC_FEATURES
    if col in X_train.columns
]

categorical_features = [
    col for col in MODEL_CATEGORICAL_FEATURES
    if col in X_train.columns
]

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

# ------------------------------------------------------------
# Preprocessing
#
# Numeric columns:
#   passed through unchanged
#
# Categorical columns:
#   converted from strings into one-hot encoded numbers
# ------------------------------------------------------------

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            "passthrough",
            numeric_features
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        )
    ]
)

# ------------------------------------------------------------
# Random Forest
# ------------------------------------------------------------

random_forest_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                max_depth=12,
                min_samples_leaf=5,
                random_state=42,
                n_jobs=-1,
                class_weight="balanced"
            )
        )
    ]
)

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------

random_forest_model.fit(
    X_train,
    y_train
)

print()
print("Random Forest trained successfully.")

# ------------------------------------------------------------
# Predict probability of decline
# ------------------------------------------------------------

rf_probability = random_forest_model.predict_proba(
    X_test
)[:, 1]

# ------------------------------------------------------------
# Standard classification metrics
# ------------------------------------------------------------

rf_prediction = (
    rf_probability >= 0.5
).astype(int)

rf_accuracy = accuracy_score(
    y_test,
    rf_prediction
)

rf_precision = precision_score(
    y_test,
    rf_prediction,
    zero_division=0
)

rf_recall = recall_score(
    y_test,
    rf_prediction,
    zero_division=0
)

# ------------------------------------------------------------
# Precision@50
# ------------------------------------------------------------

rf_precision_at_50 = precision_at_k(
    y_test,
    rf_probability,
    k=50
)

rf_top50_declining = int(
    round(rf_precision_at_50 * 50)
)

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print()
print("RANDOM FOREST RESULTS")
print("=" * 50)

print("Accuracy:", round(rf_accuracy, 4))
print("Precision:", round(rf_precision, 4))
print("Recall:", round(rf_recall, 4))
print("Precision@50:", round(rf_precision_at_50, 4))

print()
print(
    f"Top 50 predicted-decline pages contain "
    f"{rf_top50_declining} actual declining pages out of 50."
)

Numeric features: 18
Categorical features: 9

Random Forest trained successfully.

RANDOM FOREST RESULTS
Accuracy: 0.5838
Precision: 0.5896
Recall: 0.61
Precision@50: 0.62

Top 50 predicted-decline pages contain 31 actual declining pages out of 50.


| Method | Precision@50 | Actual declining pages in Top 50 |
|---|---:|---:|
| Week-5 baseline | 0.30 | 15 / 50 |
| Logistic Regression | **0.74** | **37 / 50** |
| Random Forest | 0.62 | 31 / 50 |

**Interpretation:** Logistic Regression performed best for our ranking objective. It identified 37 of the 50 highest-priority pages as actually declining, compared with 31 for Random Forest and 15 for the Week-5 baseline. This suggests the simpler Logistic Regression model currently provides more useful ranking performance than the more complex Random Forest on the held-out client groups.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## 4. Errors and interpretation

The Logistic Regression model achieved the strongest Precision@50, but it was not correct on every selected page. Its main errors are false positives: pages ranked highly as likely to be declining that were not labeled as declining in the observed data. This means the model should be treated as a prioritization tool rather than a final decision-maker.

The model uses page visibility, engagement, freshness, content characteristics, and search-related features to estimate decline risk. These signals are directionally reasonable because pages with different levels of visibility, age, engagement, and search performance can have different probabilities of decline.

The Random Forest did not improve the ranking result. Logistic Regression achieved 0.74 Precision@50 compared with 0.62 for Random Forest, so the added model complexity did not provide additional value for this test split.

Some high-ranked pages may still be difficult to classify because observed decline can depend on factors not represented in the starter dataset, such as changes in search demand, SERP presentation, content quality, or other external factors. Therefore, model predictions should support human review rather than automatically trigger a content refresh.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.